In [2]:
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import chromadb
from langchain.agents.structured_output import ProviderStrategy
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from typing import Literal, Optional
from pydantic import BaseModel, Field



parser = StrOutputParser()


sns.set_theme(style='whitegrid')
import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")


c:\Users\Playdata\OneDrive\Desktop\프로젝트1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)
from sentence_transformers import SentenceTransformer

emb_model = SentenceTransformer(
    "SamilPwC-AXNode-GenAI/PwC-Embedding_expr"
)
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
embed = HuggingFaceEmbeddings(
    model_name="SamilPwC-AXNode-GenAI/PwC-Embedding_expr",
    encode_kwargs={
        "normalize_embeddings": True
    }
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 7271.45it/s]


In [ ]:
books=pd.read_csv('./정제데이터/books.csv')

array(['건강/취미', '경제경영', '과학', '사회과학', '소설/시/희곡', '어린이', '에세이', '여행', '역사',
       '예술/대중문화', '요리/살림', '인문학', '자기계발', '청소년', '컴퓨터/모바일'], dtype=object)

In [33]:
class Filter(BaseModel):
    query: str = Field(
        description="책의 내용이나 특징을 검색하기 위한 자연어 검색 문장"
    )
    k: int = Field(default=5, ge=1,
        description="검색할 책의 개수. 기본값은 5")
    
    category_name: str | None = Field( default=None,
        description="책 카테고리. 조건이 없으면 None")
    
    author: str | None = Field( default=None,
        description="작가 이름. 조건이 없으면 None")
    
    min_price: int | None = Field( default=None, ge=0,
        description="최소 가격. '10000원 이상'이면 10000")
    
    max_price: int | None = Field(default=None,ge=0,
        description="최대 가격. '20000원 이하'이면 20000")
    
    min_rating: int | None = Field(default=None,ge=1,le=10,
        description="최소 평점. '평점 7 이상'이면 7")
    
    max_rating: int | None = Field(default=None,ge=1,le=10,
        description="최대 평점. '평점 8 이하'이면 8")

def filterd(query):
    """질문으로 메타데이터 필터 추출

    Args:
        query (_type_): 질문
    """
    st_model = model.with_structured_output(Filter)
    filters = st_model.invoke(query)
    data = filters.model_dump(exclude_none=True)
    equal_fields = ["category_name", "author"]
    range_fields = {
        "min_price": ("priceStandard", "$gte"),
        "max_price": ("priceStandard", "$lte"),
        "min_rating": ("customerReviewRank", "$gte"),
        "max_rating": ("customerReviewRank", "$lte"),
    }

    conditions = []
    for key, value in data.items():
        if key in equal_fields:
            conditions.append({
                key: {
                    "$eq": value
                }
            })
        elif key in range_fields:
            field, operator = range_fields[key]
            conditions.append({
                field: {
                    operator: value
                }
            })
    if not conditions:
        return {}, filters.k
    if len(conditions) == 1:
        return conditions[0], filters.k
    return {"$and": conditions}, filters.k


store = Chroma(persist_directory='chroma_db',
                    collection_name='book',
                    embedding_function=embed)

def search_chroma(query):
    """책이나 문서의 내용에 대한 질문을 검색한다.
    의미 기반 검색이나 문서 근거가 필요한 질문에 사용한다.
    단순한 수치 계산, 집계, 정렬, 조건 조회는 관계형 DB 도구를 사용한다.
    근거 문서를 찾지 못했다면 못찾았다고 말한다.
    Args:
        query (질문): _description_
    """
    where, k=filterd(query)
    search_kwargs = {"k":k }
    if where:
        search_kwargs["filter"] = where

    retriever = store.as_retriever(search_kwargs=search_kwargs)
    print(where)
    return retriever.invoke(query)


In [ ]:
search_agent=create_agent(
    model, [search_chroma],
    system_prompt=('너는 친정한 도서관 사서이다. 반드시 search_chroma 로 찾은 내용에만 '
                   '근거해 답하고, 찾은 내용이 없으면 모른다고 답하라.  형식으로 붙여라.')
)
query='별점8점이상의 우울함을 없애줄 책 추천'
result=search_agent.invoke({'messages':query})


{'customerReviewRank': {'$gte': 8}}


In [39]:
from langchain_core.runnables import RunnableLambda
search_run = RunnableLambda(search_chroma)
prompt = ChatPromptTemplate.from_messages([
    ("system",
    """
        너는 친절한 도서관 사서이다.
        반드시 검색 결과에 근거해서만 답변하라.
        검색 결과가 없으면 '검색 결과를 찾지 못했습니다'라고 답하라.
        책 제목, 작가, 가격을 함께 보여줘라.
        """),
    (
        "human",
        """
        사용자 질문:
        {query}
        """
    )
])
chain= search_run | prompt | model | parser


In [40]:
chain.invoke('1000원 이상의 재밋는 책 5개 추천해줘')


{'priceStandard': {'$gte': 1000}}


'다음은 검색 결과입니다:\n\n1. **고전 명문장 필사 100 - 생각을 깊게 삶을 단단하게**\n   - **작가**: 김지수 (엮은이)\n   - **가격**: 21,000원\n   - **출판사**: 마음시선\n   - **출간일**: 2025-04-28\n\n2. **알로하, 나의 엄마들 (반양장)**\n   - **작가**: 이금이\n   - **가격**: 15,000원\n   - **출판사**: 창비\n   - **출간일**: 2020-03-25\n\n3. **오독의 발견**\n   - **작가**: 김민철\n   - **가격**: 18,800원\n   - **출판사**: 김영사\n   - **출간일**: 2026-04-30\n\n4. **불편한 편의점 (벚꽃 에디션)**\n   - **작가**: 김호연\n   - **가격**: 16,800원\n   - **출판사**: 나무옆의자\n   - **출간일**: 2021-04-20\n\n5. **나로 향하는 길 - 열두 밤의 책방 여행**\n   - **작가**: 김슬기\n   - **가격**: 21,000원\n   - **출판사**: 책구름\n   - **출간일**: 2023-11-07\n\n필요한 정보가 더 있으면 말씀해 주세요!'

In [ ]:
system_prompt="""너는 친절한 도서관 사서이다.
        반드시 검색 결과에 근거해서만 답변하라.
        검색 결과가 없으면 '검색 결과를 찾지 못했습니다'라고 답하라.
        책 제목, 작가, 가격을 함께 보여줘라."""
def build_chain():
    """프롬프트 · 모델 · 출력 파서를 이어 붙인 체인을 만들어 돌려줍니다.

    만드는 데 시간이 드는 준비물이라 **매 메시지마다 새로 만들 필요가 없습니다**.
    이 함수는 캐싱하지 않습니다. 캐싱은 이 함수를 부르는 쪽이 정합니다
    (Streamlit 앱이라면 `@st.cache_resource`, 일반 스크립트라면 변수에 담아 두기).

    Raises:
        RuntimeError: OPENAI_API_KEY 가 없을 때.
    """


    from langchain_core.output_parsers import StrOutputParser
    from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
    from langchain_openai import ChatOpenAI

    prompt = ChatPromptTemplate.from_messages(
        [
            ("system", system_prompt),
            MessagesPlaceholder("history"),
            ("human", "{query}"),
        ]
    )
    return search_run | prompt | model | StrOutputParser()

In [ ]:
#스트림릿에서
# from core import chatbot_core
# from core.keys import require_openai_key_or_stop

# def get_chain():
#     """LangChain 체인을 만들어 돌려준다. 앱이 도는 동안 한 번만 실행된다."""
#     return chatbot_core.build_chain()
def get_chain():
    return chain

chain = get_chain()

if "messages" not in st.session_state:
    st.session_state.messages = []
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])
if prompt := st.chat_input("무엇이든 물어보세요"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.write(prompt)
    with st.chat_message("assistant"):
        # 이력에서 방금 넣은 내 메시지는 뺀다. 그 문장은 첫 번째 인자로 이미 전달된다.
        answer = st.write_stream(
            chatbot_core.stream_reply(prompt, st.session_state.messages[:-1], chain=chain)
        )
    st.session_state.messages.append({"role": "assistant", "content": answer})

ModuleNotFoundError: No module named 'core'

In [ ]:
class Filter(BaseModel):
    query: str = Field(
        description="책의 내용이나 특징을 검색하기 위한 자연어 검색 문장"
    )
    k: int = Field(default=5, ge=1,
        description="검색할 책의 개수. 기본값은 5")
    
    category_name: Literal[
    "건강/취미", "경제경영", "과학", "사회과학",
    "소설/시/희곡", "어린이", "에세이", "여행",
    "역사", "예술/대중문화", "요리/살림", "인문학",
    "자기계발", "청소년", "컴퓨터/모바일"
] | None = Field( default=None,
        description="책 카테고리. 앞의 단어가 없으면 NULL값으로 무조건 해줘")
    
    author: str | None = Field( default=None,
        description="작가 이름. 조건이 없으면 NULL값으로 무조건 해줘")
    
    min_price: int | None = Field( default=None, ge=0,
        description="최소 가격. '10000원 이상'이면 10000,숫자가 없으면 NULL값으로 무조건 해줘")
    
    max_price: int | None = Field(default=None,ge=0,
        description="최대 가격. '20000원 이하'이면 20000,숫자가 없으면 NULL값으로 무조건 해줘")
    
    min_rating: int | None = Field(default=None,ge=1,le=10,
        description="최소 평점. '평점 7 이상'이면 7,점수가 없으면 NULL값으로 무조건 해줘")
    
    max_rating: int | None = Field(default=None,ge=1,le=10,
        description="최대 평점. '평점 8 이하'이면 8,점수가 없으면 NULL값으로 무조건 해줘")

def filterd(query):
    """질문으로 메타데이터 필터 추출 질문에 조건이 없다 싶으면 무조건 뽑지마. 그리고 마음대로 무작위로 조건을 넣지마

    Args:
        query (_type_): 질문
    """
    filter_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        사용자의 질문에서 도서 검색 조건을 추출하라.

        개수 규칙:
        - '하나', '한 권', '1개'라고 하면 k=1
        - '두 권', '2개'라고 하면 k=2
        - 개수가 없으면 k=5

        카테고리 규칙:
        - 공식 카테고리명이 질문에 명시된 경우에만 category_name에 넣어라.
        - '영화', '공포', '로맨스' 같은 장르는 category_name으로 임의 추론하지 마라.
        - 공식 카테고리가 없으면 category_name=None

        가격, 평점, 작가도 질문에 명시된 경우에만 추출하라.
        언급되지 않은 조건은 모두 None으로 설정하라.
        """
        ),
        ("human", "{query}")
    ])

    filter_chain = filter_prompt | model.with_structured_output(Filter)
    print("FILTER INPUT:", repr(query))
    filters = filter_chain.invoke(query)
    print("FILTER OUTPUT:", filters)
    data = filters.model_dump(exclude_none=True)
    equal_fields = ["category_name", "author"]
    range_fields = {
        "min_price": ("priceStandard", "$gte"),
        "max_price": ("priceStandard", "$lte"),
        "min_rating": ("customerReviewRank", "$gte"),
        "max_rating": ("customerReviewRank", "$lte"),
    }

    conditions = []
    for key, value in data.items():
        if key in equal_fields:
            conditions.append({
                key: {
                    "$eq": value
                }
            })
        elif key in range_fields:
            field, operator = range_fields[key]
            conditions.append({
                field: {
                    operator: value
                }
            })
    if not conditions:
        return {}, filters.k
    if len(conditions) == 1:
        return conditions[0], filters.k
    return {"$and": conditions}, filters.k

